In [1]:
import networkx as nx
import pandas as pd
import json
import os
from typing import Any, Dict, Optional, Tuple
from helper import *
import json

def df_to_json(df: pd.DataFrame, output_path: str) -> None:
    """
    Convert the DataFrame containing influence data into a D3.js compatible JSON.
    
    Parameters:
        df: DataFrame with columns 'Name', 'Role', 'Work_influenced_by', 'Sailor_work', 'Date', 'Score'
        output_path: where to save the JSON file
    """
    # Build nodes list: sailor (fixed) + unique influencers
    nodes = [{"id": "sailor", "type": "sailor"}]
    nodes += [{"id": name, "type": "influencer"} for name in df['name'].unique()]
    
    # Build links list: one per row
    links = df.apply(lambda row: {
        "source": row['name'],
        "target": "sailor",
        "year": row['date'],         
        "role": row['role'],
        "work_influencer": row['work_influenced_by'],
        "sailor_work": row['sailor_work'],
        "score": row['score']
    }, axis=1).tolist()
    
    # Write to json
    with open(output_path +"data_task1-1.json", "w") as f:
        json.dump({"nodes": nodes, "links": links}, f, indent=2) 
    
def explore_sailor_influences(G: nx.MultiDiGraph, sailor_node):
    """ Find all people who influences Sailor """
    person_info = {} # person node -> set of (role, work_node)
    influenced_by_works = {} #sailor work -> set of works that influenced her work
    influential_persons = {}  
    
    def add_person_info(person, role, work):
        """Helper to record a person's role and associated work."""
        if person not in person_info:
            person_info[person] = set()   
        person_info[person].add((role, work))
        
    def add_influenced_by_work(sailor_work, influenced_by_work, etype):
        if sailor_work not in influenced_by_works:
            influenced_by_works[sailor_work] = set()
        influenced_by_works[sailor_work].add((influenced_by_work, etype))

    def find_role_of_person(G, person_node):
        roles = set()
        for _, _, data in G.out_edges(person_node, data=True):
            edge_type = data.get('Edge Type')
            if edge_type in person_work_types:
                roles.add(edge_type)
        return roles
        
    # --- Step 1 : find all sailor works      
    sailor_works, band_members = find_all_artist_works(G, sailor_node)
            
    # --- Step 2: Pepole who influenced Sailor with collaborating with her directly in her works.
    # for ctype in person_work_types:
    #     for sailor_work in sailor_works.keys():
    #         sailor_roles = sailor_works.get(sailor_work)
    #         for neighbor in get_neighbors_by_edge_type(G, sailor_work, ctype, 'in'):
    #             node_type = G.nodes[neighbor].get('Node Type')
    #             if node_type == 'Person':
    #                 if neighbor != sailor_node:
    #                      if neighbor and neighbor not in influential_persons:
    #                          influential_persons[neighbor] = set()
    #                      if ctype in sailor_roles:
    #                          influential_persons[neighbor].add((ctype, sailor_work, sailor_work, 1))
    #                      else:
    #                          influential_persons[neighbor].add((ctype, sailor_work, sailor_work, 1/4))
    #             elif node_type == 'MusicalGroup':
    #                 for creator in get_neighbors_by_edge_type(G, neighbor, member_type, 'in'):
    #                     if creator != sailor_node:
    #                         if creator and creator not in influential_persons:
    #                             influential_persons[creator] = set()
    #                         if ctype in sailor_roles:
    #                             influential_persons[creator].add((ctype, sailor_work, sailor_work, 1))
    #                         else:
    #                             influential_persons[creator].add((ctype, sailor_work, sailor_work, 1/4))
                         
                        
    
    #--- Step 3: Pepole who influenced Sailor with their works(InStyleOf, CoverOf,....)
    for etype in influence_work_types:
        for sailor_work in sailor_works.keys():
            sailor_work_name = G.nodes[sailor_work].get('name')
            sailor_work_release_date = G.nodes[sailor_work].get('release_date')
            sailor_roles = sailor_works.get(sailor_work)
            for neighbor in get_neighbors_by_edge_type(G, sailor_work, etype, 'out'):
                neighbor_name = G.nodes[neighbor].get('name')
                node_type = G.nodes[neighbor].get('Node Type')
                work_influenced_by_release_date = G.nodes[neighbor].get('release_date')
                if node_type in ('Song', 'Album'):
                    if int(sailor_work_release_date) > int(work_influenced_by_release_date):
                        add_influenced_by_work(sailor_work, neighbor, etype)
                        if etype == 'InterpolatesFrom': ## In this situation, only the composer of the work can influence Sailor work.
                            for creator in get_neighbors_by_edge_type(G, neighbor, 'ComposerOf', 'in'):
                                if creator != sailor_node:
                                    node_type_creator = G.nodes[creator].get('Node Type')
                                    if node_type_creator == 'Person':
                                        if creator and creator not in influential_persons:
                                            influential_persons[creator] = set() 
                                        if 'ComposerOf' in sailor_roles:
                                           influential_persons[creator].add(('ComposerOf', neighbor_name, sailor_work_name, sailor_work_release_date, 1))
                                        else:  
                                           influential_persons[creator].add(('ComposerOf', neighbor_name, sailor_work_name, sailor_work_release_date, 1/2))
                                    if node_type_creator == 'MusicalGroup':
                                        for creator_member in get_neighbors_by_edge_type(G, creator, member_type, 'in'):
                                            if creator_member != sailor_node:
                                                if creator_member not in influential_persons:
                                                    influential_persons[creator_member] = set()
                                                if 'ComposerOf' in sailor_roles:
                                                    influential_persons[creator_member].add(('ComposerOf', neighbor_name, sailor_work_name, sailor_work_release_date, 1))
                                                else:
                                                    influential_persons[creator_member].add(('ComposerOf', neighbor_name, sailor_work_name, sailor_work_release_date, 1/2))
                                    
                        else: ##if etype == InStyleOf, CoverOf, LyricalReferenceTo, or DirectlySamples
                            for ctype in person_work_types:
                                for creator in get_neighbors_by_edge_type(G, neighbor, ctype , 'in'):
                                    node_type_creator = G.nodes[creator].get('Node Type')
                                    if node_type_creator == 'Person':
                                        if creator and creator not in influential_persons:
                                            influential_persons[creator] = set()
                                        if ctype in sailor_roles and creator:
                                            influential_persons[creator].add((ctype, neighbor_name, sailor_work_name, sailor_work_release_date, 1))
                                        else:
                                            influential_persons[creator].add((ctype, neighbor_name, sailor_work_name, sailor_work_release_date,1/2))
                                    if node_type_creator == 'MusicalGroup':
                                            for creator_member in get_neighbors_by_edge_type(G, creator, member_type, 'in'):
                                                if creator_member != sailor_node:
                                                    if creator_member not in influential_persons:
                                                        influential_persons[creator_member] = set()
                                                    if ctype in sailor_roles:
                                                        influential_persons[creator_member].add((ctype, neighbor_name, sailor_work_name, sailor_work_release_date,1))
                                                    else:
                                                        influential_persons[creator_member].add((ctype, neighbor_name, sailor_work_name, sailor_work_release_date,1/2))
                elif node_type in ('Person'):
                    set_of_roles_of_person = find_role_of_person(G, neighbor)
                    for person_role in set_of_roles_of_person:
                        if person_role in sailor_roles:
                            if neighbor not in influential_persons:
                                influential_persons[neighbor] = set()
                            influential_persons[neighbor].add((person_role, 'null', sailor_work_name, sailor_work_release_date, 1))
                        else: # person_role != sailor_role
                            if neighbor not in influential_persons:
                                influential_persons[neighbor] = set()
                            influential_persons[neighbor].add((person_role, 'null', sailor_work_name, sailor_work_release_date , 1/2))  
                            
                elif node_type in ('MusicalGroup'):
                    for member in get_neighbors_by_edge_type(G, neighbor, member_type, 'in'):
                        if member not in influential_persons:
                            influential_persons[member] = set()
                        influential_persons[member].add(('PerformerOf', 'null', sailor_work_name, sailor_work_release_date , 1))
          
        
    return sailor_works, band_members,influential_persons,person_info            
                    
                                   
def main():
    G = read_data_from_json_to_graph(data_file_path)
    n, a = find_sailor_shift(G) 
    print(n, a)
    sailor_works, band_members, influential_persons, person_info = explore_sailor_influences(G, 17255)
    rows = []
    for sailor_work in sailor_works:
        genre = G.nodes[sailor_work].get('genre')
        print(f"genre:{genre}")
    # print(influential_persons)
    # print(f"{sailor_works} and{len(sailor_works)} and *****important****** :{influential_persons}")
    for key, tuple_set in influential_persons.items():
        influence_name = G.nodes[key].get('name')
        roles = []
        for tup in tuple_set:
            rows.append({
                'name' : influence_name,
                'role' : tup[0],
                'work_influenced_by' : tup[1],
                'sailor_work' : tup[2],
                'date': tup[3],
                'score' : tup[4]
                 })
            
    df = pd.DataFrame(rows)
    df_to_json(df, 'data/')
    print(df)
    # df['total_score'] = df.groupby('name')['score'].transform('sum')
    # pd.set_option('display.max_rows', None)   
    
    # grouped = df.groupby('Name')['Score'].sum().reset_index()
    # print(grouped)
 
        
if __name__ == "__main__":
    main()

Type of data: <class 'dict'>
17255 {'Node Type': 'Person', 'name': 'Sailor Shift'}
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Synthwave
genre:Oceanus Folk
genre:Americana
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
genre:Oceanus Folk
                 name         role    work_influe